In [0]:
# =============================================================================
# NOTEBOOK  : silver_customers
# PURPOSE   : bronze.customers → silver.customers
# LAYER     : Silver (clean, type-cast, parse, merge)
# FREQUENCY : Daily (Autoloader availableNow)
# TRIGGER   : availableNow
#
# TRANSFORM:
#   - first_purchase_date  : String → DateType
#   - total_spend          : Double → Decimal(10,2)
#   - preferred_categories : raw JSON string → ArrayType(StringType)
#   - loyalty_tier         : trim and title-case for consistency
#
# MERGE & DEDUP LOGIC:
#   Dedup   : dropDuplicates on customer_id before merge
#             daily CSV could have duplicate customer records
#   Case 1  : customer_id NOT in silver → INSERT
#   Case 2  : customer_id IN silver, data changed → UPDATE
#             Changeable: age_group, gender, zip_code, loyalty_tier,
#                         total_spend, preferred_categories
#             first_purchase_date NOT updated — immutable, set once on first purchase
#   Case 3  : customer_id IN silver, no change → IGNORE
# =============================================================================

# ── CELL 1: Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")

from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_CUSTOMERS_SCHEMA

from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, trim, initcap,
    from_json, regexp_replace
)
from pyspark.sql.types import DecimalType, ArrayType, StringType
from delta.tables import DeltaTable

with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)

SOURCE_TABLE = cfg["tables"]["bronze_customers"]
TARGET_TABLE = cfg["tables"]["silver_customers"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_customers"]
PIPELINE     = "silver_customers"

In [0]:
# ── foreachBatch function + Stream ────────────────────────────────────

def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch of bronze customer rows.

    DEDUP  : dropDuplicates on customer_id before merge
             daily CSV may contain duplicate customer records
             assumption: duplicates are exactly identical

    MERGE  :
      Case 1 → customer_id not in silver → INSERT
      Case 2 → customer_id in silver, data changed → UPDATE
      Case 3 → customer_id in silver, no change → IGNORE

    NOTE   : first_purchase_date is never updated
             it is set once when customer first appears — immutable
    """

    # ── executor-side imports ─────────────────────────────────
    import sys
    sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
    from utils.audit import start_audit, end_audit
    from utils.schema_registry import SILVER_CUSTOMERS_SCHEMA

    if micro_batch_df.isEmpty():
        return

    run_id = start_audit(spark, PIPELINE, TARGET_TABLE, SOURCE_TABLE)

    try:
        rows_read = micro_batch_df.count()

        # ── TRANSFORM ─────────────────────────────────────────────────────────

        df = (
            micro_batch_df

            # 1. first_purchase_date: String → DateType
            #    Source format: "2017-01-10 00:00:00" or "2017-01-10"
            #    to_date auto-detects both formats
            .withColumn(
                "first_purchase_date",
                to_date(col("first_purchase_date"))
            )

            # 2. total_spend: Double → Decimal(10,2)
            .withColumn(
                "total_spend",
                col("total_spend").cast(DecimalType(10, 2))
            )

            # 3. preferred_categories: raw JSON array string → ArrayType(StringType)
            #    Source: '["Books", "Food & Beverage", "Clothing"]'
            #    Strip any leading/trailing whitespace before parsing
            .withColumn(
                "preferred_categories",
                from_json(
                    trim(col("preferred_categories")),
                    ArrayType(StringType())
                )
            )

            # 4. loyalty_tier: trim whitespace + title-case for consistency
            #    Source may have "PLATINUM", "platinum", " Gold " etc.
            #    Standardize to: Bronze, Silver, Gold, Platinum
            .withColumn(
                "loyalty_tier",
                initcap(trim(col("loyalty_tier")))
            )

            # 5. Silver audit columns
            .withColumn("processed_at",    current_timestamp())
            .withColumn("pipeline_run_id", lit(run_id))
            .withColumn("source_system",   lit("customers_csv"))

            # 6. Enforce silver schema — drops all bronze-only columns
            .select([f.name for f in SILVER_CUSTOMERS_SCHEMA.fields])
        )

        # ── DEDUP WITHIN MICRO-BATCH ──────────────────────────────────────────
        # Drop duplicate customer_ids before merge
        # Delta MERGE does not deduplicate source df internally
        # Two rows with same customer_id = two inserts/updates
        df = df.dropDuplicates(["customer_id"])

        rows_after_dedup = df.count()
        if rows_read != rows_after_dedup:
            print(f"[DEDUP] dropped {rows_read - rows_after_dedup} "
                  f"duplicate customer rows in microbatch_id={microbatch_id}")

        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # Case 2: matched + data changed → UPDATE
        #         first_purchase_date intentionally excluded from update
        #         it is immutable — set once when customer first appears
        # Case 1: not matched → INSERT
        # Case 3: matched + no change → no rule fires → IGNORE
        (
            DeltaTable.forName(spark, TARGET_TABLE).alias("t")
            .merge(
                df.alias("s"),
                "t.customer_id = s.customer_id"
            )
            .whenMatchedUpdate(
                condition="""
                    t.age_group             != s.age_group              OR
                    t.gender                != s.gender                 OR
                    t.zip_code              != s.zip_code               OR
                    t.loyalty_tier          != s.loyalty_tier           OR
                    t.total_spend           != s.total_spend            OR
                    t.preferred_categories  != s.preferred_categories
                """,
                set={
                    "age_group":            "s.age_group",
                    "gender":               "s.gender",
                    "zip_code":             "s.zip_code",
                    "loyalty_tier":         "s.loyalty_tier",
                    "total_spend":          "s.total_spend",
                    "preferred_categories": "s.preferred_categories",
                    "processed_at":         "current_timestamp()",
                    "pipeline_run_id":      f"'{run_id}'"
                    # first_purchase_date → NOT updated, immutable
                    # source_system       → NOT updated, preserve original
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))

        end_audit(
            spark, run_id, "SUCCESS",
            rows_read=rows_read,
            rows_written=rows_inserted + rows_updated,
            extra={
                "rows_inserted": str(rows_inserted),
                "rows_updated":  str(rows_updated),
                "microbatch_id": str(microbatch_id)
            }

        )

    except Exception as e:
        end_audit(spark, run_id, "FAILED", error=str(e))
        raise


# ── READ BRONZE AS STREAM ─────────────────────────────────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table(SOURCE_TABLE)
)

# ── WRITE WITH foreachBatch ───────────────────────────────────────────────────
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col

print("=" * 60)
print("AUDIT LOG — Last 3 runs")
print("=" * 60)
(
    spark.read.table("azure3_team7_project.platform.pipeline_audit")
    .filter(col("pipeline_name") == PIPELINE)
    .orderBy(col("start_time").desc())
    .limit(3)
    .select(
        "run_id", "status", "rows_read",
        "rows_written", "start_time", "error_message"
    )
    .show(truncate=False)
)

print("=" * 60)
print("SILVER TABLE — Row count and sample")
print("=" * 60)
silver_df = spark.read.table(TARGET_TABLE)
print(f"Total rows: {silver_df.count()}")

print("=" * 60)
print("SILVER TABLE — loyalty_tier distribution")
print("=" * 60)
(
    silver_df
    .groupBy("loyalty_tier")
    .count()
    .orderBy("loyalty_tier")
    .show(truncate=False)
)

print("=" * 60)
print("SILVER TABLE — Sample rows with preferred_categories parsed")
print("=" * 60)
(
    silver_df
    .select(
        "customer_id", "loyalty_tier",
        "total_spend", "preferred_categories",
        "first_purchase_date", "source_system"
    )
    .limit(5)
    .show(truncate=False)
)

print("=" * 60)
print("DELTA HISTORY — Last 3 operations")
print("=" * 60)
(
    spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 3")
    .select("version", "timestamp", "operation", "operationMetrics")
    .show(truncate=False)
)